In [2]:
import pandas as pd
from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "Churn"

test_ids = test["id"]

train = train.drop(columns=["id"])
test = test.drop(columns=["id"])

y = train[TARGET]
X = train.drop(columns=[TARGET])


In [ ]:
import pandas as pd
from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "Churn"

test_ids = test["id"]

train = train.drop(columns=["id"])
test = test.drop(columns=["id"])

y = train[TARGET]
X = train.drop(columns=[TARGET])

# find categorical columns
cat_features = X.select_dtypes("object").columns.tolist()

model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    eval_metric="AUC",
    loss_function="Logloss",
    task_type="GPU",
    devices="0",
    random_seed=42,
    early_stopping_rounds=200
)

model.fit(
    X,
    y,
    cat_features=cat_features,
    verbose=200
)

pred_probs = model.predict_proba(test)[:,1]

submission = pd.DataFrame({
    "id": test_ids,
    "Churn": pred_probs
})

submission.to_csv("submission_catboost_gpu.csv", index=False)

## Catboost Optimization

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "Churn"

test_ids = test["id"]

train = train.drop(columns=["id"])
test = test.drop(columns=["id"])


y = train[TARGET]
X = train.drop(columns=[TARGET])

cat_features = X.select_dtypes("object").columns.tolist()


X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.03,
    depth=8,

    eval_metric="AUC",
    loss_function="Logloss",

    task_type="GPU",
    devices="0",

    random_seed=42,

    early_stopping_rounds=200,

    verbose=200
)


model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    cat_features=cat_features
)


pred_probs = model.predict_proba(test)[:, 1]


submission = pd.DataFrame({
    "id": test_ids,
    "Churn": pred_probs
})

submission.to_csv("submission_catboost_gpuV1.1.csv", index=False)

print("submission_catboost_gpuV1.1.csv created")

## Catboost with Stratifier K-fold Cross Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
cat_features = X.select_dtypes(include=["object","string"]).columns.tolist()


N_FOLDS = 5

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))


for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"\n===== Fold {fold+1} =====")

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    model = CatBoostClassifier(

        iterations=5000,
        learning_rate=0.03,
        depth=8,

        eval_metric="AUC",
        loss_function="Logloss",

        task_type="GPU",
        devices="0",

        random_seed=42,

        early_stopping_rounds=200,

        verbose=200
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        cat_features=cat_features
    )

    val_pred = model.predict_proba(X_val)[:,1]
    oof_preds[val_idx] = val_pred

    test_preds += model.predict_proba(test)[:,1] / N_FOLDS

y_binary = y.map({"No":0,"Yes":1})
auc = roc_auc_score(y_binary, oof_preds)

print("\nCV ROC-AUC:", auc)

submission = pd.DataFrame({
    "id": test_ids,
    "Churn": test_preds
})

submission.to_csv("submission_catboost_cvV1.2.csv", index=False)

print("\nsubmission_catboost_cvV1.2.csv created")

## Catboost with Feature Engineering

In [ ]:

from sklearn.metrics import roc_auc_score

#Feature Engineering
train["avg_monthly_spend"] = train["TotalCharges"] / (train["tenure"] + 1)
test["avg_monthly_spend"] = test["TotalCharges"] / (test["tenure"] + 1)

train["tenure_group"] = pd.cut(
    train["tenure"],
    bins=[0,12,24,48,60,100],
    labels=["0-1yr","1-2yr","2-4yr","4-5yr","5+yr"]
)

test["tenure_group"] = pd.cut(
    test["tenure"],
    bins=[0,12,24,48,60,100],
    labels=["0-1yr","1-2yr","2-4yr","4-5yr","5+yr"]
)

service_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

train["num_services"] = (train[service_cols] == "Yes").sum(axis=1)
test["num_services"] = (test[service_cols] == "Yes").sum(axis=1)

y = train[TARGET]
X = train.drop(columns=[TARGET])

cat_features = X.select_dtypes(
    include=["object","string","category"]
).columns.tolist()


N_FOLDS = 5

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))


for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"\n===== Fold {fold+1} =====")

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    model = CatBoostClassifier(

        iterations=5000,
        learning_rate=0.03,
        depth=8,

        loss_function="Logloss",
        eval_metric="AUC",

        task_type="GPU",
        devices="0",

        random_seed=42,

        early_stopping_rounds=200,

        verbose=200
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        cat_features=cat_features
    )

    val_pred = model.predict_proba(X_val)[:,1]
    oof_preds[val_idx] = val_pred

    test_preds += model.predict_proba(test)[:,1] / N_FOLDS


y_binary = y.map({"No":0,"Yes":1})

cv_auc = roc_auc_score(y_binary, oof_preds)

print("\nCV ROC-AUC:", cv_auc)


submission = pd.DataFrame({
    "id": test_ids,
    "Churn": test_preds
})

submission.to_csv("submission_catboost_cv_FEv1.csv", index=False)

print("\nsubmission_catboost_cv_fe.csv created")

## Catbost and Optuna Optimized

In [5]:

import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
train["avg_monthly_spend"] = train["TotalCharges"] / (train["tenure"] + 1)
test["avg_monthly_spend"] = test["TotalCharges"] / (test["tenure"] + 1)

train["tenure_group"] = pd.cut(
    train["tenure"],
    bins=[0,12,24,48,60,100],
    labels=["0-1yr","1-2yr","2-4yr","4-5yr","5+yr"]
)

test["tenure_group"] = pd.cut(
    test["tenure"],
    bins=[0,12,24,48,60,100],
    labels=["0-1yr","1-2yr","2-4yr","4-5yr","5+yr"]
)

service_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

train["num_services"] = (train[service_cols] == "Yes").sum(axis=1)
test["num_services"] = (test[service_cols] == "Yes").sum(axis=1)


y = train[TARGET]
X = train.drop(columns=[TARGET])

cat_features = X.select_dtypes(
    include=["object","string","category"]
).columns.tolist()


def objective(trial):

    params = {

        "iterations": 3000,

        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),

        "depth": trial.suggest_int("depth", 6, 10),

        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),

        "bagging_temperature": trial.suggest_float(
            "bagging_temperature", 0, 1
        ),

        "random_strength": trial.suggest_float(
            "random_strength", 0, 1
        ),

        "loss_function": "Logloss",

        "eval_metric": "AUC",

        "task_type": "GPU",

        "devices": "0",

        "verbose": 0
    }

    skf = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    auc_scores = []

    for train_idx, val_idx in skf.split(X, y):

        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]

        model = CatBoostClassifier(**params)

        model.fit(
            X_train,
            y_train,
            eval_set=(X_val, y_val),
            cat_features=cat_features,
            early_stopping_rounds=100,
            verbose=False
        )

        preds = model.predict_proba(X_val)[:,1]

        y_val_binary = y_val.map({"No":0,"Yes":1})

        auc = roc_auc_score(y_val_binary, preds)

        auc_scores.append(auc)

    return np.mean(auc_scores)


study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=20)

print("\nBest AUC:", study.best_value)

print("\nBest parameters:")
print(study.best_params)

[I 2026-03-11 00:07:14,422] A new study created in memory with name: no-name-a11911eb-cd07-46a6-aa40-ede7e9d926d8
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-03-11 00:22:35,862] Trial 0 finished with value: 0.9157692018156925 and parameters: {'learning_rate': 0.01807202691810856, 'depth': 10, 'l2_leaf_reg': 7.916982125593238, 'bagging_temperature': 0.14561304178451817, 'random_strength': 0.6246702170546687}. Best is trial 0 with value: 0.9157692018156925.
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-03-11 00:37:19,187] Trial 1 finished with value: 0.9161741788139167 and parameters: {'learning_rate': 0.019450270279707145, 'depth': 7, 'l2_leaf_reg


Best AUC: 0.9162248939377354

Best parameters:
{'learning_rate': 0.03776503308766629, 'depth': 6, 'l2_leaf_reg': 3.6338224225352977, 'bagging_temperature': 0.5885913474160037, 'random_strength': 0.7737520245715626}


In [6]:
best_params = study.best_params

final_model = CatBoostClassifier(

    iterations=5000,

    learning_rate=best_params["learning_rate"],
    depth=best_params["depth"],
    l2_leaf_reg=best_params["l2_leaf_reg"],
    bagging_temperature=best_params["bagging_temperature"],
    random_strength=best_params["random_strength"],

    loss_function="Logloss",
    eval_metric="AUC",

    task_type="GPU",
    devices="0",

    verbose=200
)

final_model.fit(
    X,
    y,
    cat_features=cat_features
)

pred_probs = final_model.predict_proba(test)[:,1]

submission = pd.DataFrame({
    "id": test_ids,
    "Churn": pred_probs
})

submission.to_csv("submission_catboost_optunaV1.csv", index=False)

print("\nSubmission file created: submission_catboost_optuna.csv")

Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 115ms	remaining: 9m 36s
200:	total: 22.8s	remaining: 9m 3s
400:	total: 45.1s	remaining: 8m 37s
600:	total: 1m 7s	remaining: 8m 14s
800:	total: 1m 30s	remaining: 7m 52s
1000:	total: 1m 52s	remaining: 7m 29s
1200:	total: 2m 14s	remaining: 7m 7s
1400:	total: 2m 37s	remaining: 6m 44s
1600:	total: 2m 59s	remaining: 6m 21s
1800:	total: 3m 22s	remaining: 5m 59s
2000:	total: 3m 45s	remaining: 5m 37s
2200:	total: 4m 7s	remaining: 5m 14s
2400:	total: 4m 30s	remaining: 4m 52s
2600:	total: 4m 52s	remaining: 4m 29s
2800:	total: 5m 14s	remaining: 4m 7s
3000:	total: 5m 37s	remaining: 3m 44s
3200:	total: 5m 59s	remaining: 3m 22s
3400:	total: 6m 22s	remaining: 2m 59s
3600:	total: 6m 45s	remaining: 2m 37s
3800:	total: 7m 7s	remaining: 2m 14s
4000:	total: 7m 30s	remaining: 1m 52s
4200:	total: 7m 53s	remaining: 1m 29s
4400:	total: 8m 15s	remaining: 1m 7s
4600:	total: 8m 38s	remaining: 44.9s
4800:	total: 9m	remaining: 22.4s
4999:	total: 9m 23s	remaining: 0us

Submission file created: submission_c